# Model Inference — Kaggle Submission

Loads the **best** model from the MLflow **Model Registry**, predicts on the **raw** test set, and writes a Kaggle submission. No manual preprocessing: the registered Pipeline handles merge + feature engineering internally.

Self-contained notebook — no imports from a shared `src/` package.

## 0. Config & environment (Kaggle vs local)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_COMPETITION = 'walmart-recruiting-store-sales-forecasting'
ON_KAGGLE = KAGGLE_INPUT.exists()

if ON_KAGGLE:
    RAW_DIR = KAGGLE_INPUT / KAGGLE_COMPETITION
    WORKING_DIR = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path.cwd().parent
    RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
    WORKING_DIR = PROJECT_ROOT
    load_dotenv(PROJECT_ROOT / '.env')

def _resolve(stem):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        p = RAW_DIR / name
        if p.exists():
            return p
    return RAW_DIR / f'{stem}.csv'

TRAIN_CSV = _resolve('train')
TEST_CSV = _resolve('test')
FEATURES_CSV = _resolve('features')
STORES_CSV = _resolve('stores')

RANDOM_SEED = 42
TARGET = 'Weekly_Sales'
HOLIDAY_WEIGHT = 5
NON_HOLIDAY_WEIGHT = 1

if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for key in ('MLFLOW_TRACKING_URI', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD'):
            try:
                os.environ.setdefault(key, client.get_secret(key))
            except Exception:
                pass
    except Exception:
        pass

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
MLFLOW_TRACKING_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME')
MLFLOW_TRACKING_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD')

print('On Kaggle:', ON_KAGGLE, '| raw data dir:', RAW_DIR)

## 1. Data loading helper (raw test only)

In [ ]:
import pandas as pd

def _read_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False})

def load_stores():
    return pd.read_csv(STORES_CSV)

def load_features():
    df = pd.read_csv(FEATURES_CSV, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    df = df.sort_values(['Store', 'Date'])
    for col in ('CPI', 'Unemployment'):
        df[col] = df.groupby('Store')[col].transform(lambda s: s.ffill().bfill())
    return df.reset_index(drop=True)

def load_raw(split):
    path = TRAIN_CSV if split == 'train' else TEST_CSV
    df = pd.read_csv(path, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    return df

def load_merged(split='train'):
    base = load_raw(split)
    stores = load_stores()
    feats = load_features().drop(columns=['IsHoliday'])
    df = base.merge(stores, on='Store', how='left')
    df = df.merge(feats, on=['Store', 'Date'], how='left')
    df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    return df

## 2. Submission builder

In [ ]:
import numpy as np, pandas as pd

def make_submission(test_df, preds, path=None, clip_negative=True):
    preds = np.asarray(preds, dtype=float)
    if len(preds) != len(test_df):
        raise ValueError(f'preds length {len(preds)} != test rows {len(test_df)}')
    if clip_negative:
        preds = np.clip(preds, 0, None)
    dates = pd.to_datetime(test_df['Date']).dt.strftime('%Y-%m-%d')
    ids = test_df['Store'].astype(str) + '_' + test_df['Dept'].astype(str) + '_' + dates
    sub = pd.DataFrame({'Id': ids.to_numpy(), 'Weekly_Sales': preds})
    out_path = path or (WORKING_DIR / 'submission.csv')
    sub.to_csv(out_path, index=False)
    print(f'Wrote {len(sub):,} rows -> {out_path}')
    return sub

## 3. MLflow tracking + load best model from Registry

In [ ]:
import mlflow

def init_tracking(experiment=None):
    uri = MLFLOW_TRACKING_URI
    if uri:
        mlflow.set_tracking_uri(uri)
        if MLFLOW_TRACKING_USERNAME:
            os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_TRACKING_USERNAME
        if MLFLOW_TRACKING_PASSWORD:
            os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_TRACKING_PASSWORD
    else:
        mlflow.set_tracking_uri((WORKING_DIR / 'mlruns').as_uri())
    if experiment:
        mlflow.set_experiment(experiment)
    return mlflow.get_tracking_uri()

init_tracking()
MODEL_NAME = 'walmart_best_model'
STAGE_OR_VERSION = 'latest'

## Load best model from the Model Registry

In [ ]:
model_uri = f'models:/{MODEL_NAME}/{STAGE_OR_VERSION}'
model = mlflow.sklearn.load_model(model_uri)
print('loaded', model_uri)

## Predict on the raw test set

In [ ]:
raw_test = load_raw('test')
preds = model.predict(raw_test)
sub = make_submission(raw_test, preds)
sub.head()

## Submit to Kaggle (optional)
Requires Kaggle credentials (KAGGLE_USERNAME / KAGGLE_KEY).